In [1]:
from langchain.agents import create_agent
from langchain_text_splitters import MarkdownHeaderTextSplitter

# =============1.准备文档================
# 1.读取文档
with open('./resources/评估.md', 'r', encoding='utf-8') as f:
    doc = f.read()

# 2.切分文档
headers_to_split_on = [
    ("#", "H1"),
    ("##", "H2"),
    ("###", "H3"),
]
markdown_splitter = MarkdownHeaderTextSplitter(
    headers_to_split_on,
    strip_headers=False # 是否去除文本中的标题，False：不去除
)
# list[Document]
documents = markdown_splitter.split_text(doc)
print(len(documents))

21


In [2]:
for i, document in enumerate(documents, start=1):
    document.id = f"doc_{i}"

In [3]:
from langchain_ollama import OllamaEmbeddings
from langchain_core.vectorstores import InMemoryVectorStore

# =============2.准备VectorStore================

# 1.初始化向量模型
ollama_embeddings = OllamaEmbeddings(
    model="qwen3-embedding:0.6b",  # 性价比高的模型
    dimensions=1024  # 向量维度，可选：减少维度以节省存储
)

# 2.初始化VectorStore
vector_store = InMemoryVectorStore(ollama_embeddings)

In [4]:
# =============3.添加和删除文档================

ids = vector_store.add_documents(documents)

In [10]:
from langchain_core.tools import tool
from langchain.chat_models import init_chat_model
from langchain.agents import create_agent
# ================Agentic RAG===================

# 1.加载环境变量
from dotenv import load_dotenv
load_dotenv()

# 2.初始化模型
model = init_chat_model(
    "deepseek-v4-flash",
    extra_body = {"thinking": {"type": "disabled"}}
)

# 3.定义工具
@tool
def retrieve_docs(query: str):
    """检索知识库，获取与用户提问相关的知识片段"""

    print("===========用户提问：===========")
    print(query)
    # 1.根据问题检索知识片段
    documents = vector_store.similarity_search(query)

    # 2.拼接知识片段
    context = "\n".join([doc.page_content for doc in documents])
    print("===========检索文档：===========")
    print(context)

    return context

# 4.定义Agent，绑定model和Tool
agent = create_agent(
    model,
    tools=[retrieve_docs],
    system_prompt="你是一个教资考试专家，回答与教资相关问题时，必须调用工具检索文档，根据文档回答用户的问题，不能自己编造答案。答案要简短，直接引用原文。"
)

In [11]:
from langchain_core.messages import HumanMessage

response = agent.invoke(
    {"messages": [HumanMessage("论语中教育的目的是什么？")]} # 查询重写
)
print("==================AI回答==================")
print(response["messages"][-1].content)

===========用户提问：===========
论语 教育目的
===========检索文档：===========
### （三）孔子及《论语》主要思想  
**教育作用**：庶、富、教；性相近，习相远。  
**教育对象**：“有教无类”，教育民主思想。  
**教育目的**：以完善人格为教育的首要目的，培养士和君子。  
**教学内容**：文、行、忠、义。  
**教学过程**：学、思、习、行。  
**教学原则**：因材施教、启发诱导、学思结合、谦虚笃实。  
**教师观**：其身正，不令而行；其身不正，虽令不从。
### （四）孟子主要思想  
**人性论**：人性本善，人先天具有仁、义、礼、智四个“善端”。  
**教育作用**：发扬善端，培养道德完人，得天下英才而教育之。  
**教学原则**：循序渐进，专心有恒。
### （五）荀子主要思想  
**人性论**：性恶论，化性起伪，善德是后天习得的，重视教育的作用。  
**教学原则**：学以致用，锲而不舍。
### （二）《学记》主要思想  
**教育作用**：化民成俗，其必由学；建国君民，教学为先。  
**教学相长**：教和学两方面互相影响和促进，都得到提高。  
**豫时孙摩**：包括预防性原则、及时性原则、循序渐进原则、集体教育原则。  
**长善救失**：“学者有四失，教者必知之。人之学也，或失则多，或失则寡，或失则易，或失则止。此四者，心之莫同也。知其心，然后能救其失也，教也者，长善而救其失者也。”  
**启发诱导**：道而弗牵，强而弗抑，开而弗达。
根据知识库文档，《论语》中关于教育目的的记载为：

**以完善人格为教育的首要目的，培养士和君子。**


In [12]:
from langchain_core.messages import HumanMessage

response = agent.invoke(
    {"messages": [HumanMessage("你好")]} # 查询重写
)
print("==================AI回答==================")
print(response["messages"][-1].content)

==================AI回答==================
你好！我是教资考试助手，有什么关于教师资格证考试的问题需要我帮忙解答吗？
